<div dir="rtl">

# جلسه ۵ — جست‌وجوی BFS و DFS (نوت‌بوک آموزشی)

> **درس:** مقدمه‌ای بر هوش مصنوعی و کاربرد آن در مهندسی پزشکی
> **همراه با جزوه:** `session_05_search_bfs_dfs.md`
> **توجه:** نقشه انتقال و همه عددهای این نوت‌بوک فرضی و صرفاً آموزشی‌اند.

## در این نوت‌بوک چه می‌کنیم؟
1. شبکه انتقال بیمار بین بخش‌های بیمارستان را به‌صورت گراف تعریف می‌کنیم.
2. الگوریتم‌های **BFS** (با صف) و **DFS** (با پشته) را پیاده‌سازی می‌کنیم.
3. اجرای هر الگوریتم را **گام‌به‌گام** می‌بینیم: لیست دیده‌شده‌ها و صف/پشته در هر گام.
4. با چند آزمایش (تغییر ترتیب همسایه‌ها، حذف یال، ماز) رفتار الگوریتم‌ها را مقایسه می‌کنیم.
5. چند تمرین حل می‌کنیم.

**نحوه استفاده:** سلول‌ها را به ترتیب اجرا کنید (**Run All**). بعد از فهم هر بخش، سراغ آزمایش‌ها و تمرین‌ها بروید.

> [!NOTE]
> خروجی‌ها به‌جای `print` به‌صورت **جدول HTML راست‌به‌چپ** نمایش داده می‌شوند، چون متن فارسی مخلوط با نام‌های لاتین (مثل `ER -> RAD`) در خروجی `print` درست نمایش داده نمی‌شود.

</div>

<div dir="rtl">

## ۰. ابزارهای نمایش

سلول زیر فقط توابع نمایش جدول‌ها را تعریف می‌کند. **آن را اجرا کنید و از آن بگذرید**؛ الگوریتم‌ها در بخش‌های بعد هستند.

</div>

In [1]:
import copy
import html
from collections import deque

from IPython.display import HTML, display

# ---- استایل جدول‌ها (در حالت روشن و تیره خوانا است) ----
_TABLE = "border-collapse:collapse;direction:rtl;text-align:center;margin:8px 0"
_CELL = "border:1px solid #888;padding:4px 10px;vertical-align:top"
_HEAD = _CELL + ";background:rgba(128,128,128,.25)"


def _e(x):
    return html.escape(str(x))


def _ltr(inner):
    """متن لاتین را چپ‌به‌راست و تک‌فاصله نمایش می‌دهد تا ترتیب عناصر به‌هم نریزد."""
    return f'<span dir="ltr" style="font-family:monospace">{inner}</span>'


def _table(headers, rows, caption=None):
    out = [f'<table style="{_TABLE}">']
    if caption:
        out.append(f'<caption style="caption-side:top;font-weight:bold">{caption}</caption>')
    out.append("<tr>" + "".join(f'<th style="{_HEAD}">{h}</th>' for h in headers) + "</tr>")
    for r in rows:
        out.append("<tr>" + "".join(f'<td style="{_CELL}">{c}</td>' for c in r) + "</tr>")
    out.append("</table>")
    display(HTML("".join(out)))


def _heading(text):
    display(HTML(f'<h4 dir="rtl" style="margin:12px 0 0 0">{text}</h4>'))


def show_graph(graph, names):
    """جدول فهرست همسایه‌ها (به ترتیب بررسی) به‌همراه وزن یال‌ها."""
    rows = []
    for node, nbrs in graph.items():
        txt = ", ".join(f"{_e(n)} ({w})" for n, w in nbrs)
        rows.append([_ltr(_e(node)), _e(names.get(node, "—")), _ltr(txt)])
    _table(["گره", "نام فارسی", "همسایه‌ها (به ترتیب بررسی) و وزن یال"], rows)


def show_maze(maze):
    rows = []
    for line in maze:
        cells = []
        for ch in line:
            label = {"S": "S (شروع)", "G": "G (هدف)", "X": "X (بسته)", ".": "باز"}[ch]
            cells.append(label)
        rows.append(cells)
    _table(["ستون ۱", "ستون ۲", "ستون ۳"][:len(maze[0])], rows)


def show_trace(log, kind):
    """جدول گام‌به‌گام. kind='queue' → صف افقی؛ kind='stack' → پشته عمودی."""
    rows = []
    for e in log:
        new = set(e["new"])
        visited = ", ".join(f"<b>{_e(v)}</b>" if v in new else _e(v) for v in e["visited"]) or "—"
        node = "—" if e["node"] is None else _ltr(_e(e["node"]))
        if e["goal"]:
            node += " ✅ هدف"
        if e["note"]:
            node += f" <small>({_e(e['note'])})</small>"
        children = _ltr(", ".join(map(_e, e["new"]))) if e["new"] else "—"
        if kind == "queue":
            frontier = _ltr("[" + ", ".join(map(_e, e["frontier"])) + "]")
        else:
            items = list(reversed(e["frontier"]))          # بالاترین = سر پشته
            lines = [f"<b>{_e(items[0])}</b>"] + [_e(x) for x in items[1:]] if items else []
            lines.append("━━━━")
            frontier = _ltr("<br>".join(lines))
        rows.append([e["step"], node, children, _ltr(visited), frontier])
    if kind == "queue":
        headers = ["گام", "خروج از صف", "فرزندان جدید", "دیده‌شده‌ها (پررنگ = تازه کشف‌شده)", "صف (چپ = سر صف)"]
    else:
        headers = ["گام", "خروج از پشته", "فرزندان جدید (ترتیب push)", "دیده‌شده‌ها (پررنگ = تازه کشف‌شده)", "پشته (بالا = سر پشته)"]
    _table(headers, rows)


def _stats(graph, path, expansions, log):
    return {
        "path": path,
        "steps": len(path) - 1,
        "cost": path_cost(graph, path),
        "popped": expansions,
        "expanded": expansions - 1,
        "seen": len(log[-1]["visited"]),
        "max_frontier": max(len(e["frontier"]) for e in log),
        "left": list(log[-1]["frontier"]),
    }


_ROWS = [
    ("مسیر یافته‌شده", lambda s: _ltr(" → ".join(map(_e, s["path"])))),
    ("تعداد گام (یال)", lambda s: s["steps"]),
    ("هزینه مسیر", lambda s: s["cost"]),
    ("گره‌های خارج‌شده از مرز (شامل هدف)", lambda s: s["popped"]),
    ("گره‌های بسط‌داده‌شده (بدون هدف)", lambda s: s["expanded"]),
    ("گره‌های دیده‌شده", lambda s: s["seen"]),
    ("بیشینه اندازه مرز", lambda s: s["max_frontier"]),
    ("باقی‌مانده در مرز هنگام توقف", lambda s: _ltr(", ".join(map(_e, s["left"]))) if s["left"] else "هیچ"),
]


def show_result(graph, path, expansions, log):
    s = _stats(graph, path, expansions, log)
    _table(["معیار", "مقدار"], [[name, f(s)] for name, f in _ROWS])


def compare_table(graph, bfs_res, dfs_res):
    sb, sd = _stats(graph, *bfs_res), _stats(graph, *dfs_res)
    _table(["معیار", "BFS", "DFS"], [[name, f(sb), f(sd)] for name, f in _ROWS])

<div dir="rtl">

## ۱. گراف شبکه انتقال بیمار

بیماری در **اورژانس** (`ER`) است و باید به **بخش بستری B** (`WARD_B`) منتقل شود. نقشه انتقال یک **گراف بی‌جهت و وزن‌دار** است:

- هر **گره** یک بخش است؛
- هر **یال** یک مسیر انتقال است و **وزن** آن زمان انتقال (دقیقه) است؛
- هر یال در دو جهت قابل پیمودن است.

گراف را با یک **دیکشنری** نگه می‌داریم: کلید = گره، مقدار = لیست `(همسایه، وزن)`.

> [!IMPORTANT]
> **ترتیب** همسایه‌ها در لیست مهم است: الگوریتم‌ها همسایه‌ها را به همین ترتیب بررسی می‌کنند و در DFS این ترتیب می‌تواند مسیر نهایی را عوض کند.

</div>

In [2]:
# گراف شبکه انتقال بیمار (بی‌جهت، وزن‌دار؛ وزن = زمان انتقال به دقیقه، داده فرضی)
GRAPH = {
    "ER":     [("RAD", 4), ("LAB", 2), ("PHARM", 1)],
    "PHARM":  [("ER", 1)],
    "RAD":    [("ER", 4), ("ICU", 5), ("LAB", 3)],
    "LAB":    [("ER", 2), ("RAD", 3), ("OR", 2)],
    "OR":     [("LAB", 2), ("ICU", 1)],
    "ICU":    [("RAD", 5), ("OR", 1), ("WARD_B", 2)],
    "WARD_B": [("ICU", 2)],
}

FA = {
    "ER": "اورژانس", "RAD": "تصویربرداری", "LAB": "آزمایشگاه",
    "OR": "اتاق عمل", "ICU": "مراقبت ویژه", "WARD_B": "بخش بستری B",
    "PHARM": "داروخانه",
}

START, GOAL = "ER", "WARD_B"

show_graph(GRAPH, FA)

گره,نام فارسی,همسایه‌ها (به ترتیب بررسی) و وزن یال
ER,اورژانس,"RAD (4), LAB (2), PHARM (1)"
PHARM,داروخانه,ER (1)
RAD,تصویربرداری,"ER (4), ICU (5), LAB (3)"
LAB,آزمایشگاه,"ER (2), RAD (3), OR (2)"
OR,اتاق عمل,"LAB (2), ICU (1)"
ICU,مراقبت ویژه,"RAD (5), OR (1), WARD_B (2)"
WARD_B,بخش بستری B,ICU (2)


<div dir="rtl">

**صورت‌بندی مسئله:**

<div dir="rtl">

| جزء | در این مثال |
|--:|--:|
| وضعیت آغازین | `ER` |
| وضعیت | بخشی که بیمار در آن است |
| کنش | رفتن به یک بخش مجاور (هر یال یک کنش) |
| آزمون هدف | وضعیت فعلی برابر `WARD_B` باشد |
| هزینه مسیر | مجموع وزن یال‌های مسیر (دقیقه) |

</div>

</div>

<div dir="rtl">

## ۲. توابع کمکی

- `reconstruct_path`: هر گره **والد** خودش را به یاد دارد (دیکشنری `came_from`). با دنبال کردن والدها از هدف تا شروع و معکوس کردن، مسیر ساخته می‌شود.
- `path_cost`: وزن یال‌های مسیر را جمع می‌کند.
- `_log`: فقط برای **نمایش** است؛ اگر لیستی به آن بدهیم، وضعیت هر گام را در آن ثبت می‌کند. الگوریتم‌ها بدون آن هم کار می‌کنند.

</div>

In [3]:
def reconstruct_path(came_from, goal):
    """بازسازی مسیر از دیکشنری والدها."""
    node, path = goal, []
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()
    return path


def path_cost(graph, path):
    """جمع وزن یال‌های مسیر."""
    cost = 0
    for u, v in zip(path, path[1:]):
        cost += dict(graph[u])[v]
    return cost


def _log(log, step, node, new, visited, frontier, goal=False, note=""):
    """ثبت وضعیت یک گام برای جدول نمایش (اگر log=None باشد کاری نمی‌کند)."""
    if log is not None:
        log.append({
            "step": step, "node": node, "new": list(new),
            "visited": list(visited), "frontier": list(frontier),
            "goal": goal, "note": note,
        })

<div dir="rtl">

## ۳. الگوی کلی جست‌وجو

BFS و DFS **یک الگوریتم‌اند** و فقط در ساختار داده‌ی **مرز** (Frontier) تفاوت دارند:

<div dir="rtl">

| | BFS | DFS |
|--:|:-:|:-:|
| مرز | صف (FIFO) | پشته (LIFO) |
| خروج گره از | سر صف (قدیمی‌ترین) | سر پشته (تازه‌ترین) |
| رفتار | سطح‌به‌سطح | عمیق رفتن و عقب‌گرد |

</div>

```text
frontier <- [start];  visited <- {start}
while frontier is not empty:
    node <- REMOVE(frontier)          # BFS: front of queue | DFS: top of stack
    if node == goal: return PATH(node)
    for each neighbor of node (in the listed order):
        if neighbor not in visited:
            visited.add(neighbor)     # marked when it ENTERS the frontier
            parent[neighbor] <- node
            ADD(frontier, neighbor)
```

**قراردادهای این نوت‌بوک (همان قراردادهای جزوه):**
1. گره هنگام **ورود** به مرز «دیده‌شده» علامت می‌خورد؛ پس هیچ گرهی دو بار وارد مرز نمی‌شود.
2. آزمون هدف هنگام **خروج** گره از مرز انجام می‌شود.
3. «گره‌های خارج‌شده از مرز» شامل گره هدف است؛ «بسط‌داده‌شده» بدون هدف و یکی کمتر است.

</div>

<div dir="rtl">

## ۴. جست‌وجوی سطح‌اول (BFS)

BFS مرز را با **صف** (`collections.deque`) مدیریت می‌کند: `popleft()` قدیمی‌ترین گره کشف‌شده را برمی‌دارد و فرزندان تازه به **ته** صف (`append`) می‌روند. نتیجه: ابتدا همه گره‌های سطح ۱، بعد سطح ۲ و … بسط داده می‌شوند؛ مثل موجی که در آب پخش می‌شود. به همین دلیل BFS **کم‌یال‌ترین** مسیر را می‌یابد.

دیکشنری `came_from` دو کار می‌کند: هم **والد** هر گره را نگه می‌دارد و هم نقش **لیست دیده‌شده‌ها** را دارد (کلیدهایش همان گره‌های دیده‌شده‌اند، به ترتیب کشف).

</div>

In [4]:
def bfs(graph, start, goal, log=None):
    """جست‌وجوی سطح‌اول: صف FIFO، کمترین تعداد گام را می‌یابد."""
    frontier = deque([start])          # صف
    came_from = {start: None}          # والدها = لیست دیده‌شده‌ها
    expansions = 0
    step = 0
    _log(log, step, None, [], came_from, frontier)
    while frontier:
        node = frontier.popleft()      # خروج از سر صف
        expansions += 1
        step += 1
        if node == goal:               # آزمون هدف هنگام خروج
            _log(log, step, node, [], came_from, frontier, goal=True)
            return reconstruct_path(came_from, goal), expansions
        new = []
        for nbr, _w in graph.get(node, []):
            if nbr not in came_from:   # هنوز دیده نشده
                came_from[nbr] = node  # علامت‌گذاری هنگام ورود به مرز
                frontier.append(nbr)   # ورود به ته صف
                new.append(nbr)
        _log(log, step, node, new, came_from, frontier)
    return None, expansions

<div dir="rtl">

### اجرای BFS و جدول گام‌به‌گام

در ستون «صف»، **سمت چپ = سر صف** (خروج از اینجا) و سمت راست = ته صف (ورود از اینجا) است.

</div>

In [5]:
log_bfs = []
path_bfs, exp_bfs = bfs(GRAPH, START, GOAL, log=log_bfs)

show_trace(log_bfs, "queue")
show_result(GRAPH, path_bfs, exp_bfs, log_bfs)

گام,خروج از صف,فرزندان جدید,دیده‌شده‌ها (پررنگ = تازه کشف‌شده),صف (چپ = سر صف)
0,—,—,ER,[ER]
1,ER,"RAD, LAB, PHARM","ER, RAD, LAB, PHARM","[RAD, LAB, PHARM]"
2,RAD,ICU,"ER, RAD, LAB, PHARM, ICU","[LAB, PHARM, ICU]"
3,LAB,OR,"ER, RAD, LAB, PHARM, ICU, OR","[PHARM, ICU, OR]"
4,PHARM,—,"ER, RAD, LAB, PHARM, ICU, OR","[ICU, OR]"
5,ICU,WARD_B,"ER, RAD, LAB, PHARM, ICU, OR, WARD_B","[OR, WARD_B]"
6,OR,—,"ER, RAD, LAB, PHARM, ICU, OR, WARD_B",[WARD_B]
7,WARD_B ✅ هدف,—,"ER, RAD, LAB, PHARM, ICU, OR, WARD_B",[]


معیار,مقدار
مسیر یافته‌شده,ER → RAD → ICU → WARD_B
تعداد گام (یال),3
هزینه مسیر,11
گره‌های خارج‌شده از مرز (شامل هدف),7
گره‌های بسط‌داده‌شده (بدون هدف),6
گره‌های دیده‌شده,7
بیشینه اندازه مرز,3
باقی‌مانده در مرز هنگام توقف,هیچ


<div dir="rtl">

**چند نکته از جدول:**
- در گام ۲، `LAB` از `RAD` هم قابل دسترسی است، اما چون قبلاً دیده شده، دوباره وارد صف **نمی‌شود**.
- `WARD_B` در گام ۵ کشف می‌شود، ولی آزمون هدف تا گام ۷ (هنگام خروج از صف) انجام نمی‌شود.
- **مسیر را از ترتیب خروج نمی‌خوانیم!** ترتیب خروج (`ER, RAD, LAB, PHARM, ICU, OR, WARD_B`) فقط ترتیب بسط است؛ مسیر از روی والدها ساخته می‌شود:

> `WARD_B ← ICU ← RAD ← ER`  ⟹  `ER → RAD → ICU → WARD_B`

</div>

<div dir="rtl">

## ۵. جست‌وجوی عمق‌اول (DFS)

DFS مرز را با **پشته** (یک لیست پایتون) مدیریت می‌کند: `pop()` **آخرین** گره واردشده را برمی‌دارد. پس تا جایی که ممکن است در عمق پیش می‌رود و وقتی به **بن‌بست** رسید، به گره‌ای که پایین‌تر در پشته منتظر مانده برمی‌گردد (عقب‌گرد).

> [!TIP]
> فرزندان به ترتیب فهرست همسایه‌ها push می‌شوند؛ پس **آخرین همسایه روی پشته قرار می‌گیرد و اولین گرهی است که بسط داده می‌شود.** در `ER` همسایه‌ها `RAD, LAB, PHARM` هستند و `PHARM` زودتر از بقیه خارج می‌شود.

</div>

In [6]:
def dfs(graph, start, goal, log=None):
    """جست‌وجوی عمق‌اول: پشته LIFO، یک شاخه را تا انتها می‌رود."""
    stack = [start]                    # پشته (آخر لیست = سر پشته)
    came_from = {start: None}
    expansions = 0
    step = 0
    _log(log, step, None, [], came_from, stack)
    while stack:
        node = stack.pop()             # خروج از سر پشته (تازه‌ترین)
        expansions += 1
        step += 1
        if node == goal:
            _log(log, step, node, [], came_from, stack, goal=True)
            return reconstruct_path(came_from, goal), expansions
        new = []
        for nbr, _w in graph.get(node, []):
            if nbr not in came_from:
                came_from[nbr] = node
                stack.append(nbr)      # push روی سر پشته
                new.append(nbr)
        _log(log, step, node, new, came_from, stack)
    return None, expansions

<div dir="rtl">

### اجرای DFS و جدول گام‌به‌گام

در ستون «پشته»، **بالاترین (پررنگ) = سر پشته** است و خط پایین، ته پشته را نشان می‌دهد.

</div>

In [7]:
log_dfs = []
path_dfs, exp_dfs = dfs(GRAPH, START, GOAL, log=log_dfs)

show_trace(log_dfs, "stack")
show_result(GRAPH, path_dfs, exp_dfs, log_dfs)

گام,خروج از پشته,فرزندان جدید (ترتیب push),دیده‌شده‌ها (پررنگ = تازه کشف‌شده),پشته (بالا = سر پشته)
0,—,—,ER,ER━━━━
1,ER,"RAD, LAB, PHARM","ER, RAD, LAB, PHARM",PHARMLABRAD━━━━
2,PHARM,—,"ER, RAD, LAB, PHARM",LABRAD━━━━
3,LAB,OR,"ER, RAD, LAB, PHARM, OR",ORRAD━━━━
4,OR,ICU,"ER, RAD, LAB, PHARM, OR, ICU",ICURAD━━━━
5,ICU,WARD_B,"ER, RAD, LAB, PHARM, OR, ICU, WARD_B",WARD_BRAD━━━━
6,WARD_B ✅ هدف,—,"ER, RAD, LAB, PHARM, OR, ICU, WARD_B",RAD━━━━


معیار,مقدار
مسیر یافته‌شده,ER → LAB → OR → ICU → WARD_B
تعداد گام (یال),4
هزینه مسیر,7
گره‌های خارج‌شده از مرز (شامل هدف),6
گره‌های بسط‌داده‌شده (بدون هدف),5
گره‌های دیده‌شده,7
بیشینه اندازه مرز,3
باقی‌مانده در مرز هنگام توقف,RAD


<div dir="rtl">

**چند نکته از جدول:**
- **گام ۲:** `PHARM` بن‌بست است (تنها همسایه‌اش `ER` قبلاً دیده شده). چیزی push نمی‌شود و گام بعد به `LAB` برمی‌گردیم؛ این همان **عقب‌گرد** است.
- **گام ۶:** هدف پیدا شد، اما `RAD` هنوز ته پشته است: **دیده شد ولی هرگز بسط داده نشد.**
- مسیر `ER → LAB → OR → ICU → WARD_B` هزینه ۷ دارد که از مسیر BFS (۱۱) ارزان‌تر است؛ اما این **تصادف** ترتیب همسایه‌هاست، نه تضمین DFS (آزمایش ۱ را ببینید).

</div>

<div dir="rtl">

## ۶. مقایسه BFS و DFS

</div>

In [8]:
compare_table(GRAPH, (path_bfs, exp_bfs, log_bfs), (path_dfs, exp_dfs, log_dfs))

معیار,BFS,DFS
مسیر یافته‌شده,ER → RAD → ICU → WARD_B,ER → LAB → OR → ICU → WARD_B
تعداد گام (یال),3,4
هزینه مسیر,11,7
گره‌های خارج‌شده از مرز (شامل هدف),7,6
گره‌های بسط‌داده‌شده (بدون هدف),6,5
گره‌های دیده‌شده,7,7
بیشینه اندازه مرز,3,3
باقی‌مانده در مرز هنگام توقف,هیچ,RAD


<div dir="rtl">

**نتیجه:** BFS مسیر کم‌یال‌تر (۳ گام) یافت ولی از نظر زمان انتقال گران‌تر بود (۱۱ در برابر ۷). هیچ‌کدام به **وزن‌ها** نگاه نکردند؛ وزن‌ها فقط در محاسبه هزینه نهایی به‌کار رفتند. در این گراف کوچک، اندازه مرز هم در هر دو یکسان است؛ تفاوت حافظه (BFS ≈ $b^d$، DFS ≈ $b \cdot m$) در گراف‌های بزرگ دیده می‌شود.

</div>

<div dir="rtl">

## ۷. آزمایش‌ها

تابع `run_experiment` هر دو الگوریتم را روی یک گراف اجرا می‌کند و جدول‌های گام‌به‌گام و مقایسه را نشان می‌دهد.

</div>

In [9]:
def run_experiment(graph, start, goal, trace=True):
    """اجرای هر دو الگوریتم؛ نمایش جدول گام‌به‌گام (اختیاری) و جدول مقایسه."""
    lb, ld = [], []
    rb = bfs(graph, start, goal, log=lb)
    rd = dfs(graph, start, goal, log=ld)
    if trace:
        _heading("BFS (صف)")
        show_trace(lb, "queue")
        _heading("DFS (پشته)")
        show_trace(ld, "stack")
    _heading("مقایسه")
    compare_table(graph, (*rb, lb), (*rd, ld))

<div dir="rtl">

### آزمایش ۱: ترتیب همسایه‌های `ER` را معکوس کنید

پیش از اجرا حدس بزنید DFS کدام مسیر را می‌یابد. (راهنمایی: حالا چه گرهی آخر push می‌شود؟)

</div>

In [10]:
g1 = copy.deepcopy(GRAPH)
g1["ER"].reverse()          # ["PHARM", "LAB", "RAD"]
print(g1["ER"])

run_experiment(g1, START, GOAL)

[('PHARM', 1), ('LAB', 2), ('RAD', 4)]


گام,خروج از صف,فرزندان جدید,دیده‌شده‌ها (پررنگ = تازه کشف‌شده),صف (چپ = سر صف)
0,—,—,ER,[ER]
1,ER,"PHARM, LAB, RAD","ER, PHARM, LAB, RAD","[PHARM, LAB, RAD]"
2,PHARM,—,"ER, PHARM, LAB, RAD","[LAB, RAD]"
3,LAB,OR,"ER, PHARM, LAB, RAD, OR","[RAD, OR]"
4,RAD,ICU,"ER, PHARM, LAB, RAD, OR, ICU","[OR, ICU]"
5,OR,—,"ER, PHARM, LAB, RAD, OR, ICU",[ICU]
6,ICU,WARD_B,"ER, PHARM, LAB, RAD, OR, ICU, WARD_B",[WARD_B]
7,WARD_B ✅ هدف,—,"ER, PHARM, LAB, RAD, OR, ICU, WARD_B",[]


گام,خروج از پشته,فرزندان جدید (ترتیب push),دیده‌شده‌ها (پررنگ = تازه کشف‌شده),پشته (بالا = سر پشته)
0,—,—,ER,ER━━━━
1,ER,"PHARM, LAB, RAD","ER, PHARM, LAB, RAD",RADLABPHARM━━━━
2,RAD,ICU,"ER, PHARM, LAB, RAD, ICU",ICULABPHARM━━━━
3,ICU,"OR, WARD_B","ER, PHARM, LAB, RAD, ICU, OR, WARD_B",WARD_BORLABPHARM━━━━
4,WARD_B ✅ هدف,—,"ER, PHARM, LAB, RAD, ICU, OR, WARD_B",ORLABPHARM━━━━


معیار,BFS,DFS
مسیر یافته‌شده,ER → RAD → ICU → WARD_B,ER → RAD → ICU → WARD_B
تعداد گام (یال),3,3
هزینه مسیر,11,11
گره‌های خارج‌شده از مرز (شامل هدف),7,4
گره‌های بسط‌داده‌شده (بدون هدف),6,3
گره‌های دیده‌شده,7,7
بیشینه اندازه مرز,3,4
باقی‌مانده در مرز هنگام توقف,هیچ,"PHARM, LAB, OR"


<div dir="rtl">

**نتیجه:** DFS اکنون `RAD` را اول بسط می‌دهد و مسیر `ER → RAD → ICU → WARD_B` با هزینه **۱۱** را می‌یابد؛ با یک تغییر ساده در ترتیب همسایه‌ها، هزینه از ۷ به ۱۱ رسید. BFS در این گراف تغییری نکرد، چون کوتاه‌ترین مسیر (۳ یال) یکتاست؛ البته ترتیب بسط گره‌ها درون هر سطح عوض شد.

</div>

<div dir="rtl">

### آزمایش ۲: یال `OR ↔ ICU` را حذف کنید (بن‌بست و عقب‌گرد)

</div>

In [11]:
def drop_edge(graph, a, b):
    """حذف یال بی‌جهت a-b از یک کپی گراف."""
    g = copy.deepcopy(graph)
    g[a] = [(n, w) for n, w in g[a] if n != b]
    g[b] = [(n, w) for n, w in g[b] if n != a]
    return g


g2 = drop_edge(GRAPH, "OR", "ICU")
run_experiment(g2, START, GOAL)

گام,خروج از صف,فرزندان جدید,دیده‌شده‌ها (پررنگ = تازه کشف‌شده),صف (چپ = سر صف)
0,—,—,ER,[ER]
1,ER,"RAD, LAB, PHARM","ER, RAD, LAB, PHARM","[RAD, LAB, PHARM]"
2,RAD,ICU,"ER, RAD, LAB, PHARM, ICU","[LAB, PHARM, ICU]"
3,LAB,OR,"ER, RAD, LAB, PHARM, ICU, OR","[PHARM, ICU, OR]"
4,PHARM,—,"ER, RAD, LAB, PHARM, ICU, OR","[ICU, OR]"
5,ICU,WARD_B,"ER, RAD, LAB, PHARM, ICU, OR, WARD_B","[OR, WARD_B]"
6,OR,—,"ER, RAD, LAB, PHARM, ICU, OR, WARD_B",[WARD_B]
7,WARD_B ✅ هدف,—,"ER, RAD, LAB, PHARM, ICU, OR, WARD_B",[]


گام,خروج از پشته,فرزندان جدید (ترتیب push),دیده‌شده‌ها (پررنگ = تازه کشف‌شده),پشته (بالا = سر پشته)
0,—,—,ER,ER━━━━
1,ER,"RAD, LAB, PHARM","ER, RAD, LAB, PHARM",PHARMLABRAD━━━━
2,PHARM,—,"ER, RAD, LAB, PHARM",LABRAD━━━━
3,LAB,OR,"ER, RAD, LAB, PHARM, OR",ORRAD━━━━
4,OR,—,"ER, RAD, LAB, PHARM, OR",RAD━━━━
5,RAD,ICU,"ER, RAD, LAB, PHARM, OR, ICU",ICU━━━━
6,ICU,WARD_B,"ER, RAD, LAB, PHARM, OR, ICU, WARD_B",WARD_B━━━━
7,WARD_B ✅ هدف,—,"ER, RAD, LAB, PHARM, OR, ICU, WARD_B",━━━━


معیار,BFS,DFS
مسیر یافته‌شده,ER → RAD → ICU → WARD_B,ER → RAD → ICU → WARD_B
تعداد گام (یال),3,3
هزینه مسیر,11,11
گره‌های خارج‌شده از مرز (شامل هدف),7,7
گره‌های بسط‌داده‌شده (بدون هدف),6,6
گره‌های دیده‌شده,7,7
بیشینه اندازه مرز,3,3
باقی‌مانده در مرز هنگام توقف,هیچ,هیچ


<div dir="rtl">

**نتیجه:** حالا `OR` یک بن‌بست است. DFS **دو بار** به بن‌بست می‌رسد (`PHARM` و `OR`) و عقب‌گرد می‌کند، سپس از `RAD` ادامه می‌دهد. چون تنها مسیر به `WARD_B` از `RAD` می‌گذرد، هر دو الگوریتم مسیر یکسانی می‌یابند.

</div>

<div dir="rtl">

### آزمایش ۳: ماز بیمارستانی (فعالیت ۱ جزوه)

نقشه ۳×۳ را به یک گراف با وزن ۱ تبدیل می‌کنیم؛ ترتیب همسایه‌ها: **راست، پایین، چپ، بالا**. هر خانه با `(ردیف,ستون)` نام‌گذاری می‌شود.

</div>

In [12]:
MAZE = [
    "S.X",
    ".X.",
    "..G",
]


def build_maze_graph(maze):
    """تبدیل ماز به گراف با وزن ۱؛ ترتیب همسایه‌ها: راست، پایین، چپ، بالا."""
    graph, start, goal = {}, None, None
    rows, cols = len(maze), len(maze[0])
    for r in range(rows):
        for c in range(cols):
            if maze[r][c] == "X":
                continue
            name = f"({r + 1},{c + 1})"
            if maze[r][c] == "S":
                start = name
            if maze[r][c] == "G":
                goal = name
            nbrs = []
            for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                rr, cc = r + dr, c + dc
                if 0 <= rr < rows and 0 <= cc < cols and maze[rr][cc] != "X":
                    nbrs.append((f"({rr + 1},{cc + 1})", 1))
            graph[name] = nbrs
    return graph, start, goal


show_maze(MAZE)
maze_graph, maze_start, maze_goal = build_maze_graph(MAZE)
run_experiment(maze_graph, maze_start, maze_goal)

ستون ۱,ستون ۲,ستون ۳
S (شروع),باز,X (بسته)
باز,X (بسته),باز
باز,باز,G (هدف)


گام,خروج از صف,فرزندان جدید,دیده‌شده‌ها (پررنگ = تازه کشف‌شده),صف (چپ = سر صف)
0,—,—,"(1,1)","[(1,1)]"
1,"(1,1)","(1,2), (2,1)","(1,1), (1,2), (2,1)","[(1,2), (2,1)]"
2,"(1,2)",—,"(1,1), (1,2), (2,1)","[(2,1)]"
3,"(2,1)","(3,1)","(1,1), (1,2), (2,1), (3,1)","[(3,1)]"
4,"(3,1)","(3,2)","(1,1), (1,2), (2,1), (3,1), (3,2)","[(3,2)]"
5,"(3,2)","(3,3)","(1,1), (1,2), (2,1), (3,1), (3,2), (3,3)","[(3,3)]"
6,"(3,3) ✅ هدف",—,"(1,1), (1,2), (2,1), (3,1), (3,2), (3,3)",[]


گام,خروج از پشته,فرزندان جدید (ترتیب push),دیده‌شده‌ها (پررنگ = تازه کشف‌شده),پشته (بالا = سر پشته)
0,—,—,"(1,1)","(1,1)━━━━"
1,"(1,1)","(1,2), (2,1)","(1,1), (1,2), (2,1)","(2,1)(1,2)━━━━"
2,"(2,1)","(3,1)","(1,1), (1,2), (2,1), (3,1)","(3,1)(1,2)━━━━"
3,"(3,1)","(3,2)","(1,1), (1,2), (2,1), (3,1), (3,2)","(3,2)(1,2)━━━━"
4,"(3,2)","(3,3)","(1,1), (1,2), (2,1), (3,1), (3,2), (3,3)","(3,3)(1,2)━━━━"
5,"(3,3) ✅ هدف",—,"(1,1), (1,2), (2,1), (3,1), (3,2), (3,3)","(1,2)━━━━"


معیار,BFS,DFS
مسیر یافته‌شده,"(1,1) → (2,1) → (3,1) → (3,2) → (3,3)","(1,1) → (2,1) → (3,1) → (3,2) → (3,3)"
تعداد گام (یال),4,4
هزینه مسیر,4,4
گره‌های خارج‌شده از مرز (شامل هدف),6,5
گره‌های بسط‌داده‌شده (بدون هدف),5,4
گره‌های دیده‌شده,6,6
بیشینه اندازه مرز,2,2
باقی‌مانده در مرز هنگام توقف,هیچ,"(1,2)"


<div dir="rtl">

**نتیجه:** هر دو مسیر یکسان `(1,1)→(2,1)→(3,1)→(3,2)→(3,3)` را می‌یابند (۴ گام)، اما BFS ۶ و DFS ۵ گره از مرز خارج می‌کند. DFS چون `(2,1)` (پایین) را بعد از `(1,2)` (راست) push کرد، اول به پایین رفت و `(1,2)` ته پشته ماند و هرگز بسط داده نشد.

</div>

<div dir="rtl">

### آزمایش ۴: اگر گره را دیر «دیده‌شده» علامت بزنیم چه می‌شود؟

در این نسخه از BFS گره **هنگام خروج** از صف علامت می‌خورد (نه هنگام ورود). به‌خاطر همین، یک گره می‌تواند چند بار وارد صف شود. در این نسخه «دیده‌شده‌ها» یعنی گره‌هایی که تاکنون از صف **خارج و بسط** شده‌اند.

</div>

In [13]:
def bfs_late_mark(graph, start, goal, log=None):
    """BFS با علامت‌گذاری هنگام خروج از صف (برای نمایش مشکل تکرار)."""
    frontier = deque([start])
    parent = {start: None}
    visited = []                          # گره‌های خارج‌شده و بسط‌داده‌شده
    pops = 0
    _log(log, 0, None, [], visited, frontier)
    while frontier:
        node = frontier.popleft()
        pops += 1
        if node in visited:               # نسخه تکراری؛ رد می‌شود
            _log(log, pops, node, [], visited, frontier, note="تکراری؛ رد شد")
            continue
        visited.append(node)
        if node == goal:
            _log(log, pops, node, [], visited, frontier, goal=True)
            return reconstruct_path(parent, goal), pops
        new = []
        for nbr, _w in graph.get(node, []):
            if nbr not in visited:
                parent.setdefault(nbr, node)
                frontier.append(nbr)
                new.append(nbr)
        _log(log, pops, node, new, visited, frontier)
    return None, pops


log_late = []
path_late, pops_late = bfs_late_mark(GRAPH, START, GOAL, log=log_late)
show_trace(log_late, "queue")
display(HTML(f'<p dir="rtl">مسیر: {_ltr(" → ".join(path_late))} — تعداد خروج از صف: <b>{pops_late}</b> '
             f'(در BFS اصلی: {exp_bfs})</p>'))

گام,خروج از صف,فرزندان جدید,دیده‌شده‌ها (پررنگ = تازه کشف‌شده),صف (چپ = سر صف)
0,—,—,—,[ER]
1,ER,"RAD, LAB, PHARM",ER,"[RAD, LAB, PHARM]"
2,RAD,"ICU, LAB","ER, RAD","[LAB, PHARM, ICU, LAB]"
3,LAB,OR,"ER, RAD, LAB","[PHARM, ICU, LAB, OR]"
4,PHARM,—,"ER, RAD, LAB, PHARM","[ICU, LAB, OR]"
5,ICU,"OR, WARD_B","ER, RAD, LAB, PHARM, ICU","[LAB, OR, OR, WARD_B]"
6,LAB (تکراری؛ رد شد),—,"ER, RAD, LAB, PHARM, ICU","[OR, OR, WARD_B]"
7,OR,—,"ER, RAD, LAB, PHARM, ICU, OR","[OR, WARD_B]"
8,OR (تکراری؛ رد شد),—,"ER, RAD, LAB, PHARM, ICU, OR",[WARD_B]
9,WARD_B ✅ هدف,—,"ER, RAD, LAB, PHARM, ICU, OR, WARD_B",[]


<div dir="rtl">

**نتیجه:** مسیر درست است، اما `LAB` و `OR` هر کدام **دو بار** وارد صف شدند و ۲ خروج اضافه (۹ به‌جای ۷) انجام شد. با علامت‌گذاری هنگام **ورود** (BFS اصلی) این کار تکراری رخ نمی‌دهد.

</div>

<div dir="rtl">

## ۸. تمرین‌ها

### تمرین ۱: آزمون هدف زودهنگام در BFS
تا اینجا آزمون هدف را هنگام **خروج** گره از صف انجام دادیم. تابع `bfs_early_goal` را طوری کامل کنید که آزمون هدف را هنگام **تولید فرزند** انجام دهد و به‌محض دیدن هدف مسیر را برگرداند. مسیر باید همان مسیر قبلی باشد، ولی تعداد گره‌های بسط‌داده‌شده کمتر شود.

> راهنمایی: در حلقه‌ی همسایه‌ها، بعد از ثبت والد، بررسی کنید `nbr == goal` است یا نه. شمارنده را همچنان هنگام خروج از صف زیاد کنید.

</div>

In [14]:
def bfs_early_goal(graph, start, goal):
    """TODO: BFS با آزمون هدف هنگام تولید فرزند. خروجی: (path, expansions)"""
    # کد خود را اینجا بنویسید
    return None, 0


# ---- بررسی خودکار ----
res_path, res_exp = bfs_early_goal(GRAPH, START, GOAL)
if res_path == ["ER", "RAD", "ICU", "WARD_B"] and res_exp == 5:
    display(HTML('<p dir="rtl">✅ درست است: همان مسیر با ۵ بسط.</p>'))
else:
    display(HTML(f'<p dir="rtl">⏳ هنوز درست نیست. خروجی فعلی: مسیر {_ltr(str(res_path))} با {res_exp} بسط. '
                 'انتظار: مسیر <code>ER → RAD → ICU → WARD_B</code> با ۵ بسط.</p>'))

<div dir="rtl">

<details>
<summary>راه‌حل تمرین ۱</summary>

```python
def bfs_early_goal(graph, start, goal):
    if start == goal:
        return [start], 0
    frontier = deque([start])
    came_from = {start: None}
    expansions = 0
    while frontier:
        node = frontier.popleft()
        expansions += 1
        for nbr, _w in graph.get(node, []):
            if nbr not in came_from:
                came_from[nbr] = node
                if nbr == goal:                       # آزمون هدف هنگام تولید
                    return reconstruct_path(came_from, goal), expansions
                frontier.append(nbr)
    return None, expansions
```

چرا ۵؟ گره‌های `ER, RAD, LAB, PHARM, ICU` بسط داده می‌شوند و هدف هنگام بسط `ICU` تولید و شناسایی می‌شود؛ `OR` دیگر بسط داده نمی‌شود.

</details>

</div>

<div dir="rtl">

### تمرین ۲: ترتیب دلخواه

ترتیب همسایه‌های `ER` را به یکی از شش جایگشت ممکن تغییر دهید. **پیش از اجرا** پیش‌بینی کنید مسیر DFS و هزینه‌اش چه می‌شود، سپس اجرا کنید. چند جایگشت هزینه ۷ و چند جایگشت هزینه ۱۱ می‌دهند؟

</div>

In [15]:
from itertools import permutations

rows = []
for order in permutations(["RAD", "LAB", "PHARM"]):
    g = copy.deepcopy(GRAPH)
    weights = dict(GRAPH["ER"])
    g["ER"] = [(n, weights[n]) for n in order]
    p, e = dfs(g, START, GOAL)
    rows.append([_ltr(", ".join(order)), _ltr(" → ".join(p)), path_cost(g, p), e])

_table(["ترتیب همسایه‌های ER", "مسیر DFS", "هزینه", "خروج از پشته"], rows)

ترتیب همسایه‌های ER,مسیر DFS,هزینه,خروج از پشته
"RAD, LAB, PHARM",ER → LAB → OR → ICU → WARD_B,7,6
"RAD, PHARM, LAB",ER → LAB → OR → ICU → WARD_B,7,5
"LAB, RAD, PHARM",ER → RAD → ICU → WARD_B,11,5
"LAB, PHARM, RAD",ER → RAD → ICU → WARD_B,11,4
"PHARM, RAD, LAB",ER → LAB → OR → ICU → WARD_B,7,5
"PHARM, LAB, RAD",ER → RAD → ICU → WARD_B,11,4


<div dir="rtl">

### تمرین ۳: پرسش‌های مفهومی
1. چرا در DFS این نوت‌بوک، `PHARM` پیش از `LAB` بسط داده شد، با این‌که در فهرست `ER` بعد از آن آمده است؟
2. چرا BFS در آزمایش ۱ مسیرش را عوض نکرد ولی DFS عوض کرد؟
3. اگر `came_from` را حذف کنیم و فقط یک لیست `visited` نگه داریم، چه چیزی از دست می‌رود؟
4. برای یافتن مسیر **ارزان‌تر** (نه کم‌یال‌تر) به چه الگوریتمی نیاز داریم؟ (راهنمایی: جلسه بعد)

</div>

<div dir="rtl">

## ۹. جمع‌بندی

- BFS و DFS در الگوریتم یکسان‌اند و فقط **ساختار مرز** (صف یا پشته) فرق دارد.
- **BFS** کم‌یال‌ترین مسیر را تضمین می‌کند ولی حافظه‌ی زیادی می‌خواهد؛ **DFS** حافظه‌ی کم می‌خواهد ولی به ترتیب همسایه‌ها وابسته است و بهینگی ندارد.
- هیچ‌کدام وزن یال‌ها را نمی‌بینند؛ برای کمینه کردن **هزینه** به الگوریتم‌های آگاه به هزینه نیاز داریم (جلسه بعد).
- لیست **دیده‌شده‌ها** جلوی چرخیدن در چرخه‌ها و کار تکراری را می‌گیرد؛ زمان علامت‌گذاری (ورود به مرز) مهم است.
- مسیر را از **والدها** بسازید، نه از ترتیب خروج گره‌ها.

> برای توضیح کامل و پرسش‌های خودآزمون به جزوه‌ی `session_05_search_bfs_dfs.md` مراجعه کنید.

</div>